In [1]:
from pyspark.sql import SparkSession

# -------------------------------
# 1. Spark Session
# -------------------------------
spark = SparkSession.builder \
    .appName("Decision_eng") \
    .master("spark://localhost:7077") \
    .config("spark.driver.host", "172.21.0.1") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.port", "41251") \
    .config("spark.blockManager.port", "41252") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

spark

26/04/26 15:50:53 WARN Utils: Your hostname, Adnane resolves to a loopback address: 127.0.1.1; using 192.168.100.36 instead (on interface wlp4s0)
26/04/26 15:50:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/adnane/miniconda3/envs/spark/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/adnane/.ivy2/cache
The jars for the packages stored in: /home/adnane/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-227931a9-9b82-44af-aded-89c73d61f161;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 261ms :: artifacts dl 6ms
	:: 

In [2]:
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import *
import json
from decision_engine import decide_activity

schema = ArrayType(StructType([
    StructField("context_id", StringType()),
    StructField("user_id", StringType()),
    StructField("created_at", StringType()),

    StructField("vision", StructType([
        StructField("timestamp", StringType()),
        StructField("objects", ArrayType(StringType())),
        StructField("scene_description", StringType()),
        StructField("confidence", DoubleType()),
        StructField("media_ref", StringType())
    ])),

    StructField("audio", StructType([
        StructField("timestamp", StringType()),
        StructField("transcript", StringType()),
        StructField("keywords", ArrayType(StringType())),
        StructField("confidence", DoubleType()),
        StructField("audio_ref", StringType())
    ])),

    StructField("location", StructType([
        StructField("timestamp", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
        StructField("place_label", StringType()),
        StructField("zone_type", StringType())
    ]))
]))

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "contextBuilder") \
    .option("startingOffsets", "earliest") \
    .load()

parsed_df = df.selectExpr("CAST(value AS STRING) as value") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select(explode(col("data")).alias("context"))



In [3]:
def process_batch(batch_df, batch_id):
    rows = batch_df.collect()

    for row in rows:
        context = row["context"].asDict(recursive=True)

        decision = decide_activity(context)

        print("\n===== DECISION ENGINE OUTPUT =====")
        print(json.dumps(decision, indent=2, ensure_ascii=False))


query = parsed_df.writeStream \
    .foreachBatch(process_batch) \
    .outputMode("append") \
    .start()

query.awaitTermination()

26/04/26 15:50:58 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-9b3d5ccd-afc0-4e9d-ad88-64b4c3445194. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/26 15:50:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/26 15:50:58 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.



===== DECISION ENGINE OUTPUT =====
{
  "decision_id": "dec_generated_id",
  "context_id": "ctx_001",
  "user_id": "user_001",
  "detected_activity": "travail_sur_ordinateur",
  "event_type": "normal_context",
  "priority": "low",
  "confidence": 0.86,
  "summary": "L'utilisateur est assis devant un ordinateur avec une bouteille sur le bureau.",
  "recommendation": "Pas de recommandation spécifique.",
  "action_required": false,
  "actions": [],
  "mongodb_payload": {
    "collection": "context_events",
    "document": {}
  },
  "vector_payload": {
    "should_index": true,
    "text": "Nous devons finaliser le pipeline VLM cette semaine.",
    "metadata": {
      "context_id": "ctx_001",
      "event_type": "normal_context"
    }
  }
}

===== DECISION ENGINE OUTPUT =====
{
  "decision_id": "dec_generated_id",
  "context_id": "ctx_001",
  "user_id": "user_001",
  "detected_activity": "travail_sur_ordinateur",
  "event_type": "normal_context",
  "priority": "low",
  "confidence": 0.86,


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/adnane/miniconda3/envs/spark/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/adnane/miniconda3/envs/spark/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/home/adnane/miniconda3/envs/spark/lib/python3.8/socket.py", line 681, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

: 